# AICE 머신러닝 연습 — 분류 18문제

시험 샘플과 비슷한 기본 작업 흐름으로 만든 연습 문제입니다. 실제 출제나 출제 범위를 보장하는 자료는 아닙니다. **딥러닝은 제외하고 머신러닝만 연습합니다.**

- CSV: `aice_classification_1122.csv`를 노트북과 같은 폴더에 두세요.
- 처음부터 차례로 실행합니다. 답안 셀은 직접 작성하며 정답 파일은 별도입니다.
- **2분할만 사용합니다.** X_train, X_valid, y_train, y_valid 이름을 일관되게 사용합니다.
- Pipeline, 사용자 정의 함수, 3분할, PCA, 군집화는 사용하지 않습니다.
- 새로 보는 도구는 각 문제의 '사용 이유'와 힌트부터 확인하세요. 모든 모델 코드는 직접 fit → predict → 평가 형태입니다.
- 정답 코드 실행 확인 환경: scikit-learn 1.8.0. 설치: `pip install numpy pandas matplotlib scikit-learn joblib ipykernel`.

## 데이터 안내
실제 고객 정보가 아닌 합성 데이터입니다. score 열은 모델 연습용 수치이며 현실의 금액·개월을 뜻하지 않습니다. ID는 제거하고, 정답은 churn(0=유지, 1=이탈)입니다. 입력에는 결측치가 있습니다.

## 전처리 안내
이번 연습은 시험식 단계별 조작을 익히기 위해 결측치 대체·인코딩을 먼저 수행합니다. 실제 성능을 엄격히 평가할 때는 분할 후 학습 데이터에서만 중앙값·최빈값·인코딩 기준을 구해 검증 데이터에 적용해야 합니다. StandardScaler는 이번에도 학습 데이터에만 fit합니다. 검증 점수로 여러 번 모델을 고르므로 이 점수를 독립적인 최종 성능이라고 해석하지 않습니다.


## 01. 데이터 불러오기

**사용 이유:** read_csv는 CSV를 데이터프레임으로 읽습니다.

**문제:** CSV를 읽어 data에 저장하고 앞 5행을 출력하세요.

<details><summary>힌트 보기</summary>

pd.read_csv(), head()

</details>

In [58]:
# 여기에 답안코드를 작성하세요.
import pandas as pd

data = pd.read_csv("aice_classification_1122.csv")

data.head(5)

,member_id,usage_score,engagement_score,support_load,payment_score,tenure_score,activity_score,satisfaction_score,plan,channel,churn
0,M00000,2.357350,-0.273093,-1.997418,2.746836,0.656493,-0.288030,-0.402676,standard,partner,0
1,M00001,-1.042851,1.525425,-4.031996,5.402933,-2.366438,1.795559,1.431053,premium,partner,0
2,M00002,0.521552,-0.727856,2.662319,-1.748153,-1.036344,-0.133587,2.869435,basic,web,1
3,M00003,-1.764493,0.538583,-1.058905,1.064350,-0.738976,0.372543,-0.035856,standard,web,0
4,M00004,-1.925511,-0.214991,0.758699,0.595106,-2.673587,-1.341766,2.004080,standard,partner,0


## 02. 데이터 구조 확인

**사용 이유:** shape는 행·열 수, info는 자료형과 결측 여부, describe는 수치형 요약을 보여줍니다.

**문제:** 행·열 수, 열별 자료형, 수치형 기술통계를 확인하세요.

<details><summary>힌트 보기</summary>

shape, info(), describe()

</details>

In [59]:
# 여기에 답안코드를 작성하세요.
# 행 열 수
print(data.shape)
# 자료형과 결측 여부
print(data.info())
# 수치형 요약
data.describe()

(1800, 11)
<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   member_id           1800 non-null   str    
 1   usage_score         1755 non-null   float64
 2   engagement_score    1755 non-null   float64
 3   support_load        1755 non-null   float64
 4   payment_score       1755 non-null   float64
 5   tenure_score        1755 non-null   float64
 6   activity_score      1755 non-null   float64
 7   satisfaction_score  1755 non-null   float64
 8   plan                1755 non-null   str    
 9   channel             1755 non-null   str    
 10  churn               1800 non-null   int64  
dtypes: float64(7), int64(1), str(3)
memory usage: 154.8 KB
None


,usage_score,engagement_score,support_load,payment_score,tenure_score,activity_score,satisfaction_score,churn
count,1755.000000,1755.000000,1755.000000,1755.000000,1755.000000,1755.000000,1755.000000,1800.000000
mean,0.019605,-0.042121,-0.478296,0.703141,0.002922,0.006335,0.242892,0.280000
std,1.498921,1.529464,1.686893,1.485717,1.419914,0.999175,1.443434,0.449124
min,-5.921832,-5.372172,-6.451521,-4.254399,-4.556366,-3.403182,-5.080262,0.000000
25%,-1.030390,-1.090584,-1.615590,-0.284972,-1.039250,-0.628114,-0.689450,0.000000
50%,-0.063377,0.091010,-0.622319,0.761975,0.031424,0.016258,0.322834,0.000000
75%,1.063321,1.053738,0.486660,1.674194,1.024882,0.654972,1.236127,1.000000
max,4.670420,5.455352,5.694132,5.913831,4.127344,3.053827,5.180058,1.000000


## 03. 결측치 확인

**사용 이유:** isna().sum()은 각 열의 결측치 수를 셉니다.

**문제:** 열별 결측치 수를 출력하세요.

<details><summary>힌트 보기</summary>

data.isna().sum()

</details>

In [60]:
# 여기에 답안코드를 작성하세요.
print(data.isna().sum())

member_id              0
usage_score           45
engagement_score      45
support_load          45
payment_score         45
tenure_score          45
activity_score        45
satisfaction_score    45
plan                  45
channel               45
churn                  0
dtype: int64


## 04. 결측치 대체

**사용 이유:** fillna는 누락값을 채웁니다. 중앙값은 수치형, 최빈값은 범주형 대체에 사용할 수 있습니다.

**문제:** 수치형 입력 열은 각 열의 중앙값, plan/channel은 각 열의 최빈값으로 대체하세요.

<details><summary>힌트 보기</summary>

median(), mode()[0], fillna()

</details>

In [61]:
# 여기에 답안코드를 작성하세요.
num_cols = ["usage_score", "engagement_score","support_load", "payment_score","tenure_score","activity_score","satisfaction_score"]
str_cols = ["plan","channel"]

for i in num_cols:
    data[i] = data[i].fillna(data[i].median())

for i in str_cols:
    data[i] = data[i].fillna(data[i].mode().iloc[0])

## 05. 불필요한 열 제거

**사용 이유:** 식별자는 고객을 구별하는 값으로, 이번 예측의 입력에서 제외합니다.

**문제:** member_id 열을 제거하고 data에 다시 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[...])

</details>

In [62]:
# 여기에 답안코드를 작성하세요.
data = data.drop(columns=["member_id"])

## 06. 원핫 인코딩

**사용 이유:** get_dummies는 범주형 값을 0/1 열로 바꿉니다.

**문제:** 범주형 열을 원핫 인코딩하여 data_encoded에 저장하세요. dtype=int를 지정하세요.

<details><summary>힌트 보기</summary>

pd.get_dummies(data, columns=[...], dtype=int)

</details>

In [63]:
# 여기에 답안코드를 작성하세요.
data_encoded = pd.get_dummies(
    data,
    columns=str_cols,
    dtype=int
)

data_encoded

,usage_score,engagement_score,support_load,payment_score,tenure_score,activity_score,satisfaction_score,churn,plan_basic,plan_premium,plan_standard,channel_app,channel_partner,channel_web
0,2.357350,-0.273093,-1.997418,2.746836,0.656493,-0.288030,-0.402676,0,0,0,1,0,1,0
1,-1.042851,1.525425,-4.031996,5.402933,-2.366438,1.795559,1.431053,0,0,1,0,0,1,0
2,0.521552,-0.727856,2.662319,-1.748153,-1.036344,-0.133587,2.869435,1,1,0,0,0,0,1
3,-1.764493,0.538583,-1.058905,1.064350,-0.738976,0.372543,-0.035856,0,0,0,1,0,0,1
4,-1.925511,-0.214991,0.758699,0.595106,-2.673587,-1.341766,2.004080,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,0.402137,2.324389,-2.447813,1.519838,-0.326513,-0.564475,-0.849450,1,0,1,0,0,0,1
1796,-2.496192,3.186082,-2.766980,1.003944,-0.576148,-0.312472,-0.754205,1,0,1,0,0,1,0
1797,-1.915449,0.794657,-0.384298,0.832045,-2.098672,0.211592,0.892790,0,0,1,0,1,0,0
1798,1.303756,-1.038107,-0.622319,-1.587693,-2.105426,0.703758,1.322366,1,1,0,0,1,0,0


## 07. 입력과 정답 분리

**사용 이유:** X는 예측에 사용할 입력, y는 맞혀야 하는 정답입니다.

**문제:** churn을 y, 나머지 열을 X에 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[정답열])로 X를 만듭니다.

</details>

In [64]:
# 여기에 답안코드를 작성하세요.
X = data_encoded.drop(columns=["churn"])
y = data_encoded["churn"]

## 08. 학습·검증 2분할

**사용 이유:** train_test_split은 데이터를 두 묶음으로 나눕니다. stratify=y는 정답 클래스 비율을 유지합니다.

**문제:** 학습 80%, 검증 20%로 분할하세요. 변수는 X_train, X_valid, y_train, y_valid, random_state=42, stratify=y를 사용하세요.

<details><summary>힌트 보기</summary>

train_test_split(X, y, ...)의 반환 순서를 확인하세요.

</details>

In [65]:
# 여기에 답안코드를 작성하세요.
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 09. 표준화

**사용 이유:** StandardScaler는 학습 데이터의 평균·표준편차를 기준으로 값을 변환합니다. 거리·계수 기반 모델에서 유용합니다.

**문제:** StandardScaler를 scaler로 만들고, 학습 데이터는 fit_transform하여 X_train_scaled, 검증 데이터는 transform하여 X_valid_scaled에 저장하세요.

<details><summary>힌트 보기</summary>

검증 데이터에는 fit_transform을 사용하지 않습니다.

</details>

In [66]:
# 여기에 답안코드를 작성하세요.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

## 10. 로지스틱 회귀 학습

**사용 이유:** LogisticRegression은 이름에 회귀가 들어가지만 분류 모델입니다. 표준화한 입력으로 학습합니다.

**문제:** LogisticRegression(max_iter=1000, random_state=42)을 lr로 만들어 학습하세요. 검증 예측은 pred_lr에 저장하세요.

<details><summary>힌트 보기</summary>

fit(학습 입력, 학습 정답), predict(검증 입력)

</details>

In [67]:
# 여기에 답안코드를 작성하세요.
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict(X_valid_scaled)

## 11. 의사결정나무 학습

**사용 이유:** DecisionTreeClassifier는 조건 분기로 클래스를 예측합니다. 트리는 표준화가 필수는 아닙니다.

**문제:** max_depth=5, min_samples_split=3, random_state=42로 dt를 만들고 원본 X_train으로 학습하세요. 검증 예측은 pred_dt에 저장하세요.

<details><summary>힌트 보기</summary>

여기서는 X_train_scaled 대신 X_train을 사용합니다.

</details>

In [68]:
# 여기에 답안코드를 작성하세요.
from sklearn.tree import DecisionTreeClassifier
# 트리는 표준화가 필수가 아님

dt = DecisionTreeClassifier(max_depth=5, min_samples_split=3, random_state=42)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_valid)

## 12. 랜덤 포레스트 학습

**사용 이유:** RandomForestClassifier는 여러 결정트리의 예측을 종합합니다. n_estimators는 트리 개수입니다.

**문제:** n_estimators=100, max_depth=8, random_state=42로 rf를 만들고 원본 입력으로 학습하세요. 검증 예측은 pred_rf에 저장하세요.

<details><summary>힌트 보기</summary>

RandomForestClassifier(...).fit(X_train, y_train)

</details>

In [69]:
# 여기에 답안코드를 작성하세요.
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_valid)

## 13. 분류 성능 비교

**사용 이유:** accuracy는 전체 정답 비율, precision은 1 예측의 정확성, recall은 실제 1을 찾은 비율, F1은 precision과 recall의 조화평균입니다.

**문제:** 세 모델의 정확도를 출력하고 rf의 precision, recall, F1을 출력하세요.

<details><summary>힌트 보기</summary>

모든 평가지표의 순서는 실제 정답, 예측값입니다.

</details>

In [70]:
# 여기에 답안코드를 작성하세요.
from sklearn.metrics import precision_score, recall_score, f1_score

print(precision_score(y_valid, pred_rf))
print(recall_score(y_valid, pred_rf))
print(f1_score(y_valid, pred_rf))

0.8813559322033898
0.5148514851485149
0.65


## 14. 혼동행렬

**사용 이유:** 혼동행렬의 행은 실제, 열은 예측입니다. 클래스 순서가 0,1이면 [[TN,FP],[FN,TP]]입니다.

**문제:** rf의 혼동행렬과 classification_report를 출력하세요.

<details><summary>힌트 보기</summary>

confusion_matrix(y_valid, pred_rf)

</details>

In [71]:
# 여기에 답안코드를 작성하세요.
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print(confusion_matrix(y_valid, pred_rf))
print(classification_report(y_valid, pred_rf))

[[252   7]
 [ 49  52]]
              precision    recall  f1-score   support

           0       0.84      0.97      0.90       259
           1       0.88      0.51      0.65       101

    accuracy                           0.84       360
   macro avg       0.86      0.74      0.78       360
weighted avg       0.85      0.84      0.83       360



## 15. 확률 예측과 ROC-AUC

**사용 이유:** predict는 0/1 클래스, predict_proba는 클래스별 확률을 반환합니다. ROC-AUC에는 확률을 넣습니다.

**문제:** rf의 클래스 1 확률을 proba_rf에 저장하고 ROC-AUC를 출력하세요.

<details><summary>힌트 보기</summary>

이번 데이터의 classes_는 [0,1]이므로 [:,1]이 클래스 1 확률입니다.

</details>

In [72]:
# 여기에 답안코드를 작성하세요.
proba_rf = rf.predict_proba(X_valid)[:,1]
print(roc_auc_score(y_valid, proba_rf))

0.9103941282159105


## 16. GridSearchCV 튜닝

**사용 이유:** GridSearchCV는 지정한 파라미터 조합을 학습 데이터 안에서 교차검증합니다. 검증 데이터는 fit에 넣지 않습니다.

**문제:** RandomForestClassifier(n_estimators=100, random_state=42)를 대상으로 max_depth=[4,8], min_samples_split=[2,5]를 탐색하세요. cv=3, scoring="accuracy", 변수는 gs_rf로 만드세요.

<details><summary>힌트 보기</summary>

best_params_는 최적 설정, best_score_는 교차검증 평균 점수입니다.

</details>

In [73]:
# 여기에 답안코드를 작성하세요.
from sklearn.model_selection import GridSearchCV

params = {
    "max_depth":[4,8],
    "min_samples_split":[2,5]
}

gs_rf = GridSearchCV(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    params,
    cv=3,
    scoring="accuracy"
)

gs_rf.fit(X_train, y_train);

## 17. 최적 모델로 검증

**사용 이유:** best_estimator_는 최적 설정으로 학습 데이터 전체에 재학습된 모델입니다. 다시 fit하지 않고 예측할 수 있습니다.

**문제:** 최적 모델을 best_model에 저장하고 검증 예측을 best_pred에 저장하세요. 정확도와 ROC-AUC를 출력하세요.

<details><summary>힌트 보기</summary>

print(gs_rf.best_estimator_)는 HTML 표시 오류를 피할 수 있습니다.

</details>

In [74]:
# 여기에 답안코드를 작성하세요.
best_model = gs_rf.best_estimator_
print(best_model)
best_pred = best_model.predict(X_valid)
# 정확도
print(precision_score(y_valid, pred_rf))
print(roc_auc_score(y_valid, proba_rf))

RandomForestClassifier(max_depth=8, random_state=42)
0.8813559322033898
0.9103941282159105


## 18. 모델 저장·불러오기

**사용 이유:** joblib은 학습된 scikit-learn 모델을 저장하고 불러옵니다. 이번 포레스트는 표준화 전의 인코딩된 입력을 받습니다.

**문제:** best_model을 aice_best_classifier.joblib로 저장하고 loaded_model로 불러온 뒤 검증 앞 5행을 예측하세요.

<details><summary>힌트 보기</summary>

joblib.dump(모델, 파일명), joblib.load(파일명). 전처리까지 자동 저장되는 것은 아닙니다.

</details>

In [75]:
# 여기에 답안코드를 작성하세요.
import joblib
joblib.dump(best_model, "aice_best_classifier.joblib")
load_model = joblib.load("aice_best_classifier.joblib")
print(load_model.predict(X_valid.iloc[:5]))

[0 0 0 0 0]
